# imports

In [3]:
import os
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from tqdm import tqdm
import seaborn as sns
from pathlib import Path

In [4]:
import lifesim
import os, sys
os.chdir("/home/kirschkobold/documents/life_internship/LIFEsim_yields")
sys.path.insert(0, os.getcwd())

In [5]:
from lifesim.util.habitable import single_habitable_zone

# check files

In [7]:
df = pd.read_csv("/home/kirschkobold/documents/life_internship/LIFEsim_yields/eta_catalogs/scaled_etaEEC_AFGK/NewRefScen_AFGK_Bryson2021Model1Hab2High_etaEEC45.txt", sep="\t", header=1)

In [ ]:
df

,Nuniverse,Rp,Porb,Mp,ep,ip,Omegap,omegap,thetap,Abond,...,Ts,Ds,Stype,RA,Dec,lGal,bGal,WDSsep,name,Unnamed: 32
0,0,0.50580,18537.06214,0.17157,0.0,0.53880,5.84114,0.11270,5.85244,0.78687,...,4295.0,11.25746,K,213.91534,19.18220,NaN,NaN,NaN,TIC459832522,NaN
1,0,0.54542,52664.28714,0.16460,0.0,0.53880,3.77618,5.58377,1.36640,0.62417,...,4295.0,11.25746,K,213.91534,19.18220,NaN,NaN,NaN,TIC459832522,NaN
2,0,0.82720,25017.49255,0.33320,0.0,1.16139,2.18455,6.07288,5.91288,0.19808,...,3902.6,20.43318,K,68.98021,16.50908,NaN,NaN,NaN,TIC245873777,NaN
3,0,0.98676,26982.62314,0.77989,0.0,1.16139,0.01860,0.91391,1.75486,0.77538,...,3902.6,20.43318,K,68.98021,16.50908,NaN,NaN,NaN,TIC245873777,NaN
4,0,1.01810,1614.44203,1.10961,0.0,2.62993,1.79833,1.36838,4.95134,0.06501,...,9712.0,2.63706,A,101.28700,-16.71574,NaN,NaN,NaN,TIC322899250,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1650997,499,2.26179,209.21191,5.36760,0.0,1.90445,0.01683,0.05801,0.19576,0.05201,...,4440.0,29.22000,K,24.58956,-73.34950,NaN,NaN,NaN,2MASSJ01382152-7320584,NaN
1650998,499,1.33143,269.22594,3.19580,0.0,1.90445,3.28676,1.41172,2.23528,0.71699,...,4440.0,29.22000,K,24.58956,-73.34950,NaN,NaN,NaN,2MASSJ01382152-7320584,NaN
1650999,499,0.60980,14.13255,0.12833,0.0,1.45006,2.83414,0.39760,2.22798,0.78583,...,4440.0,17.93000,K,135.38214,-65.44452,NaN,NaN,NaN,UPMJ0901-6526,NaN
1651000,499,1.34041,17.70602,5.25741,0.0,1.45006,2.44838,4.76500,4.80476,0.58273,...,4440.0,17.93000,K,135.38214,-65.44452,NaN,NaN,NaN,UPMJ0901-6526,NaN


In [ ]:
df = df.rename(columns={
        'ap': 'semimajor_p',
        'Rp': 'radius_p',
        'Ts': 'temp_s',
        'Rs': 'radius_s',
        'Stype': 'stype',
        'name': 'name_s',
        'Nuniverse': 'nuniverse',
    })

In [ ]:
(df['s_in'],
df['s_out'],
df['l_sun'],
df['hz_in'],
df['hz_out'],
df['hz_center']) = single_habitable_zone(
model='POST-MS',
temp_s=df['temp_s'].to_numpy(),
radius_s=df['radius_s'].to_numpy()
)

In [ ]:
# Habitable flag: same 4 conditions as original
df['habitable'] = np.logical_and.reduce((
    (df['semimajor_p'] > df['hz_in']).to_numpy(),
    (df['semimajor_p'] < df['hz_out']).to_numpy(),
    (df['radius_p'] >= 0.8 * (df['semimajor_p'] / np.sqrt(df['l_sun'])) ** (-0.5)).to_numpy(),
    (df['radius_p'] <= 1.4).to_numpy()
))

n_universes = len(df['nuniverse'].unique())

eta_all = df['habitable'].sum() / len(df['name_s'].unique()) / n_universes
eta_fgk = (df.loc[df['stype'] != 'M', 'habitable'].sum()
            / len(df.loc[df['stype'] != 'M', 'name_s'].unique())
            / n_universes)
eta_m   = (df.loc[df['stype'] == 'M', 'habitable'].sum()
            / len(df.loc[df['stype'] == 'M', 'name_s'].unique())
            / n_universes)

/tmp/ipykernel_2491/49230338.py:15: RuntimeWarning: invalid value encountered in scalar divide
  eta_m   = (df.loc[df['stype'] == 'M', 'habitable'].sum()


In [ ]:
eta_fgk, eta_m, eta_all

(np.float64(0.44771693735498835),
 np.float64(nan),
 np.float64(0.44771693735498835))

# check all files

In [ ]:
import pandas as pd
import numpy as np

suffixes = ["05", "15", "25", "35", "45", "55"]
base_path = "/home/kirschkobold/documents/life_internship/LIFEsim_yields/eta_catalogs/scaled_etaEEC_AFGK/"
results = []

for suffix in suffixes:
    file_path = f"{base_path}NewRefScen_AFGK_Bryson2021Model1Hab2High_etaEEC{suffix}.txt"

    df = pd.read_csv(file_path, sep="\t", header=1)
    df = df.rename(columns={
        'ap': 'semimajor_p',
        'Rp': 'radius_p',
        'Ts': 'temp_s',
        'Rs': 'radius_s',
        'Stype': 'stype',
        'name': 'name_s',
        'Nuniverse': 'nuniverse',
    })

    (df['s_in'],
     df['s_out'],
     df['l_sun'],
     df['hz_in'],
     df['hz_out'],
     df['hz_center']) = single_habitable_zone(
        model='POST-MS',
        temp_s=df['temp_s'].to_numpy(),
        radius_s=df['radius_s'].to_numpy()
    )

    df['habitable'] = np.logical_and.reduce((
        (df['semimajor_p'] > df['hz_in']).to_numpy(),
        (df['semimajor_p'] < df['hz_out']).to_numpy(),
        (df['radius_p'] >= 0.8 * (df['semimajor_p'] / np.sqrt(df['l_sun'])) ** (-0.5)).to_numpy(),
        (df['radius_p'] <= 1.4).to_numpy()
    ))

    n_universes = len(df['nuniverse'].unique())

    eta_all = df['habitable'].sum() / len(df['name_s'].unique()) / n_universes
    eta_fgk = (df.loc[df['stype'] != 'M', 'habitable'].sum()
               / len(df.loc[df['stype'] != 'M', 'name_s'].unique())
               / n_universes)

    results.append({
        "eec_suffix": suffix,
        "eta_fgk": eta_fgk,
    })

summary_table = pd.DataFrame(results)
print(summary_table)

  eec_suffix   eta_fgk
0         05  0.049304
1         15  0.150135
2         25  0.248121
3         35  0.348760
4         45  0.447717
5         55  0.548433


In [ ]:
import pandas as pd
import numpy as np

suffixes = ["05", "15", "25", "35", "45", "55"]
base_path = "/home/kirschkobold/documents/life_internship/LIFEsim_yields/eta_catalogs/scaled_etaEEC_AFGK/"
results = []

for suffix in suffixes:
    file_path = f"{base_path}NewRefScen_AFGK_Bryson2021Model1Hab2High_etaEEC{suffix}.txt"

    df = pd.read_csv(file_path, sep="\t", header=1)
    df = df.rename(columns={
        'ap': 'semimajor_p',
        'Rp': 'radius_p',
        'Ts': 'temp_s',
        'Rs': 'radius_s',
        'Stype': 'stype',
        'name': 'name_s',
        'Nuniverse': 'nuniverse',
    })

    (df['s_in'],
     df['s_out'],
     df['l_sun'],
     df['hz_in'],
     df['hz_out'],
     df['hz_center']) = single_habitable_zone(
        model='MS',
        temp_s=df['temp_s'].to_numpy(),
        radius_s=df['radius_s'].to_numpy()
    )

    df['habitable'] = np.logical_and.reduce((
        (df['semimajor_p'] > df['hz_in']).to_numpy(),
        (df['semimajor_p'] < df['hz_out']).to_numpy(),
        (df['radius_p'] >= 0.8 * (df['semimajor_p'] / np.sqrt(df['l_sun'])) ** (-0.5)).to_numpy(),
        (df['radius_p'] <= 1.4).to_numpy()
    ))

    n_universes = len(df['nuniverse'].unique())

    eta_all = df['habitable'].sum() / len(df['name_s'].unique()) / n_universes
    eta_fgk = (df.loc[df['stype'] != 'M', 'habitable'].sum()
               / len(df.loc[df['stype'] != 'M', 'name_s'].unique())
               / n_universes)

    results.append({
        "eec_suffix": suffix,
        "eta_fgk": eta_fgk,
    })

summary_table = pd.DataFrame(results)
print(summary_table)

  eec_suffix   eta_fgk
0         05  0.059689
1         15  0.181312
2         25  0.299225
3         35  0.421021
4         45  0.539481
5         55  0.660720


In [ ]:
import pandas as pd
import numpy as np

suffixes = ["05", "15", "25", "35", "45", "55"]
base_path = "/home/kirschkobold/documents/life_internship/LIFEsim_yields/eta_catalogs/scaled_etaEEC_AFGK/"
results = []

for suffix in suffixes:
    file_path = f"{base_path}NewRefScen_AFGK_Bryson2021Model1Hab2High_etaEEC{suffix}.txt"

    df = pd.read_csv(file_path, sep="\t", header=1)
    df = df.rename(columns={
        'ap': 'semimajor_p',
        'Rp': 'radius_p',
        'Ts': 'temp_s',
        'Rs': 'radius_s',
        'Stype': 'stype',
        'name': 'name_s',
        'Nuniverse': 'nuniverse',
    })

    (df['s_in'],
     df['s_out'],
     df['l_sun'],
     df['hz_in'],
     df['hz_out'],
     df['hz_center']) = single_habitable_zone(
        model='Kopparapu-Conservative',
        temp_s=df['temp_s'].to_numpy(),
        radius_s=df['radius_s'].to_numpy()
    )

    df['habitable'] = np.logical_and.reduce((
        (df['semimajor_p'] > df['hz_in']).to_numpy(),
        (df['semimajor_p'] < df['hz_out']).to_numpy(),
        (df['radius_p'] >= 0.8 * (df['semimajor_p'] / np.sqrt(df['l_sun'])) ** (-0.5)).to_numpy(),
        (df['radius_p'] <= 1.4).to_numpy()
    ))

    n_universes = len(df['nuniverse'].unique())

    eta_all = df['habitable'].sum() / len(df['name_s'].unique()) / n_universes
    eta_fgk = (df.loc[df['stype'] != 'M', 'habitable'].sum()
               / len(df.loc[df['stype'] != 'M', 'name_s'].unique())
               / n_universes)

    results.append({
        "eec_suffix": suffix,
        "eta_fgk": eta_fgk,
    })

summary_table = pd.DataFrame(results)
print(summary_table)

  eec_suffix   eta_fgk
0         05  0.045094
1         15  0.136842
2         25  0.226352
3         35  0.318020
4         45  0.407645
5         55  0.499821


In [ ]:
import pandas as pd
import numpy as np

suffixes = ["05", "15", "25", "35", "45", "55"]
base_path = "/home/kirschkobold/documents/life_internship/LIFEsim_yields/eta_catalogs/scaled_etaEEC_AFGK/"
results = []

for suffix in suffixes:
    file_path = f"{base_path}NewRefScen_AFGK_Bryson2021Model1Hab2High_etaEEC{suffix}.txt"

    df = pd.read_csv(file_path, sep="\t", header=1)
    df = df.rename(columns={
        'ap': 'semimajor_p',
        'Rp': 'radius_p',
        'Ts': 'temp_s',
        'Rs': 'radius_s',
        'Stype': 'stype',
        'name': 'name_s',
        'Nuniverse': 'nuniverse',
    })

    (df['s_in'],
     df['s_out'],
     df['l_sun'],
     df['hz_in'],
     df['hz_out'],
     df['hz_center']) = single_habitable_zone(
        model='Kopparapu-Optimistic',
        temp_s=df['temp_s'].to_numpy(),
        radius_s=df['radius_s'].to_numpy()
    )

    df['habitable'] = np.logical_and.reduce((
        (df['semimajor_p'] > df['hz_in']).to_numpy(),
        (df['semimajor_p'] < df['hz_out']).to_numpy(),
        (df['radius_p'] >= 0.8 * (df['semimajor_p'] / np.sqrt(df['l_sun'])) ** (-0.5)).to_numpy(),
        (df['radius_p'] <= 1.4).to_numpy()
    ))

    n_universes = len(df['nuniverse'].unique())

    eta_all = df['habitable'].sum() / len(df['name_s'].unique()) / n_universes
    eta_fgk = (df.loc[df['stype'] != 'M', 'habitable'].sum()
               / len(df.loc[df['stype'] != 'M', 'name_s'].unique())
               / n_universes)

    results.append({
        "eec_suffix": suffix,
        "eta_fgk": eta_fgk,
    })

summary_table = pd.DataFrame(results)
print(summary_table)

  eec_suffix   eta_fgk
0         05  0.061628
1         15  0.186921
2         25  0.308618
3         35  0.434309
4         45  0.556434
5         55  0.682021


# star stats

In [9]:
import pandas as pd
import numpy as np

suffixes = ["05", "15", "25", "35", "45", "55"]
base_path = "/home/kirschkobold/documents/life_internship/LIFEsim_yields/eta_catalogs/scaled_etaEEC_AFGK/"
results = []

for suffix in suffixes:
    file_path = f"{base_path}NewRefScen_AFGK_Bryson2021Model1Hab2High_etaEEC{suffix}.txt"

    df = pd.read_csv(file_path, sep="\t", header=1)
    df = df.rename(columns={
        'ap': 'semimajor_p',
        'Rp': 'radius_p',
        'Ts': 'temp_s',
        'Rs': 'radius_s',
        'Stype': 'stype',
        'name': 'name_s',
        'Nuniverse': 'nuuniverse',
    })

    # Print temperature distribution for this specific file/df
    print(f"--- Suffix {suffix} ---")
    print(df['temp_s'].describe())
    print()

    # Optionally, collect summary stats into results for a combined table later
    stats = df['temp_s'].describe()
    stats['suffix'] = suffix
    results.append(stats)

summary_table = pd.DataFrame(results)
print(summary_table)

--- Suffix 05 ---
count    183194.000000
mean       5840.241904
std        1310.633518
min        2600.965400
25%        4918.000000
50%        5710.000000
75%        6363.000000
max       10209.000000
Name: temp_s, dtype: float64

--- Suffix 15 ---
count    552110.000000
mean       5837.038844
std        1307.288337
min        2600.965400
25%        4918.000000
50%        5708.700000
75%        6364.000000
max       10209.000000
Name: temp_s, dtype: float64

--- Suffix 25 ---
count    915937.000000
mean       5840.878149
std        1311.902820
min        2600.965400
25%        4917.635420
50%        5709.000000
75%        6366.000000
max       10209.000000
Name: temp_s, dtype: float64

--- Suffix 35 ---
count    1.284608e+06
mean     5.838291e+03
std      1.309165e+03
min      2.600965e+03
25%      4.918000e+03
50%      5.708700e+03
75%      6.364000e+03
max      1.020900e+04
Name: temp_s, dtype: float64

--- Suffix 45 ---
count    1.651002e+06
mean     5.840355e+03
std      1.309519e

In [11]:
import pandas as pd
import numpy as np

suffixes = ["05", "15", "25", "35", "45", "55"]
base_path = "/home/kirschkobold/documents/life_internship/LIFEsim_yields/eta_catalogs/scaled_etaEEC_AFGK/"
results = []

for suffix in suffixes:
    file_path = f"{base_path}NewRefScen_AFGK_Bryson2021Model1Hab2High_etaEEC{suffix}.txt"

    df = pd.read_csv(file_path, sep="\t", header=1)
    df = df.rename(columns={
        'ap': 'semimajor_p',
        'Rp': 'radius_p',
        'Ts': 'temp_s',
        'Rs': 'radius_s',
        'Stype': 'stype',
        'name': 'name_s',
        'Nuniverse': 'nuuniverse',
    })

    # Print radius distribution for this specific file/df
    # print(f"--- Suffix {suffix} ---")
    # print(df['radius_p'].describe())
    # print()

    # Optionally, collect summary stats into results for a combined table later
    stats = df['radius_p'].describe()
    stats['suffix'] = suffix
    results.append(stats)

summary_table = pd.DataFrame(results)
print(summary_table)

              count      mean       std  min       25%      50%       75%  \
radius_p   183194.0  1.197324  0.556568  0.5  0.717225  1.05608  1.597037   
radius_p   552110.0  1.195957  0.555284  0.5  0.717342  1.05389  1.595168   
radius_p   915937.0  1.196571  0.555942  0.5  0.716970  1.05513  1.596490   
radius_p  1284608.0  1.196952  0.555830  0.5  0.717810  1.05473  1.597430   
radius_p  1651002.0  1.196403  0.556217  0.5  0.716720  1.05364  1.597160   
radius_p  2020740.0  1.196260  0.555692  0.5  0.716750  1.05396  1.596402   

              max suffix  
radius_p  2.49998     05  
radius_p  2.49998     15  
radius_p  2.50000     25  
radius_p  2.49999     35  
radius_p  2.50000     45  
radius_p  2.50000     55  


In [12]:
suffixes = ["05", "15", "25", "35", "45", "55"]
base_path = "/home/kirschkobold/documents/life_internship/LIFEsim_yields/eta_catalogs/scaled_etaEEC_AFGK/"

radius_p_max = 1.5
radius_p_min = 0.5
temp_s_max = 4369.0
temp_s_min = 3320.0

results = []

for suffix in suffixes:
    file_path = f"{base_path}NewRefScen_AFGK_Bryson2021Model1Hab2High_etaEEC{suffix}.txt"

    df = pd.read_csv(file_path, sep="\t", header=1)
    df = df.rename(columns={
        'ap': 'semimajor_p',
        'Rp': 'radius_p',
        'Ts': 'temp_s',
        'Rs': 'radius_s',
        'Stype': 'stype',
        'name': 'name_s',
        'Nuniverse': 'nuniverse',
    })

    mask = (
        (df['radius_p'] >= radius_p_min) &
        (df['radius_p'] <= radius_p_max) &
        (df['temp_s'] >= temp_s_min) &
        (df['temp_s'] <= temp_s_max)
    )

    count = mask.sum()
    results.append({'suffix': suffix, 'number': count})

summary_table = pd.DataFrame(results)
print(summary_table)

  suffix  number
0     05   14024
1     15   42580
2     25   70483
3     35   98479
4     45  126381
5     55  154868


In [13]:
suffixes = ["05", "15", "25", "35", "45", "55"]
base_path = "/home/kirschkobold/documents/life_internship/LIFEsim_yields/eta_catalogs/scaled_etaEEC_AFGK/"

radius_p_max = 1.5
radius_p_min = 0.5
temp_s_max = 7310.0
temp_s_min = 4370.0

results = []

for suffix in suffixes:
    file_path = f"{base_path}NewRefScen_AFGK_Bryson2021Model1Hab2High_etaEEC{suffix}.txt"

    df = pd.read_csv(file_path, sep="\t", header=1)
    df = df.rename(columns={
        'ap': 'semimajor_p',
        'Rp': 'radius_p',
        'Ts': 'temp_s',
        'Rs': 'radius_s',
        'Stype': 'stype',
        'name': 'name_s',
        'Nuniverse': 'nuniverse',
    })

    mask = (
        (df['radius_p'] >= radius_p_min) &
        (df['radius_p'] <= radius_p_max) &
        (df['temp_s'] >= temp_s_min) &
        (df['temp_s'] <= temp_s_max)
    )

    count = mask.sum()
    results.append({'suffix': suffix, 'number': count})

summary_table = pd.DataFrame(results)
print(summary_table)

  suffix   number
0     05   101675
1     15   306471
2     25   508038
3     35   712124
4     45   915858
5     55  1120605


In [14]:
suffixes = ["05", "15", "25", "35", "45", "55"]
base_path = "/home/kirschkobold/documents/life_internship/LIFEsim_yields/eta_catalogs/scaled_etaEEC_AFGK/"

radius_p_max = 1.5
radius_p_min = 0.5
temp_s_max = 7310.0
temp_s_min = 4370.0

results = []

for suffix in suffixes:
    file_path = f"{base_path}NewRefScen_AFGK_Bryson2021Model1Hab2High_etaEEC{suffix}.txt"

    df = pd.read_csv(file_path, sep="\t", header=1)
    print(f"Length of Suffix {suffix}: {len(df)}")


Length of Suffix 05: 183194
Length of Suffix 15: 552110
Length of Suffix 25: 915937
Length of Suffix 35: 1284608
Length of Suffix 45: 1651002
Length of Suffix 55: 2020740
